In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.table import Table
from astropy.coordinates import SkyCoord
from scipy import optimize
import emcee
from isochrones import get_ichrone
from isochrones.mist import MIST_Isochrone

PyMultiNest not imported.  MultiNest fits will not work.


In [2]:
mist = MIST_Isochrone()
iso = mist.isochrone(age=9.66, feh=0.0)
iso.head()
iso.columns

Index(['eep', 'age', 'feh', 'mass', 'initial_mass', 'radius', 'density',
       'logTeff', 'Teff', 'logg', 'logL', 'Mbol', 'delta_nu', 'nu_max',
       'phase', 'dm_deep', 'J_mag', 'H_mag', 'K_mag', 'G_mag', 'BP_mag',
       'RP_mag', 'W1_mag', 'W2_mag', 'W3_mag', 'TESS_mag', 'Kepler_mag'],
      dtype='object')

In [3]:
idx = np.argmin(np.abs(iso["initial_mass"] - 1.0))
sun_model = iso.iloc[idx]
sun_model[["initial_mass", "Teff", "logg"]]

initial_mass       1.000754
Teff            5849.706168
logg               4.415984
Name: 355, dtype: float64

In [4]:
sun_model[["G_mag", "BP_mag", "RP_mag", "J_mag", "K_mag"]]

G_mag     4.542871
BP_mag    4.845925
RP_mag    4.074199
J_mag     3.570845
K_mag     3.215686
Name: 355, dtype: float64

In [5]:
# Known stars
known_stars = {
    "Sun": {
        "Teff": 5772, 
        "FeH": 0.0,
        "photometry": {
            "G": 4.545,
            "BP": 4.849,
            "RP": 4.076,
            "J": 3.572,
            "H": 3.251,
            "K": 3.216
        }
    },
    "HD209458": {
        "Teff": 6070,
        "FeH": 0.02,
        "photometry": {
            "G": 7.65,
            "BP": 7.73,
            "RP": 7.52,
            "J": 6.59,
            "H": 6.38,
            "K": 6.29
        }
    },
    "APCol": {
        "Teff": 3300,
        "FeH": 0.0,
        "photometry": {
            "G": 12.8,
            "BP": 13.2,
            "RP": 12.0,
            "J": 8.6,
            "H": 8.0,
            "K": 7.7
        }
    }
}

In [6]:
from scipy.optimize import minimize

def chi2_star(age_log, star, mist_iso):
    # Compute chi² comparing observed star to a MIST isochrone at given age.
    
    Teff_obs = star["Teff"]
    FeH_obs = star["FeH"]
    phot_obs = star["photometry"]

    iso = mist_iso.isochrone(age=age_log, feh=FeH_obs)

    idx = np.argmin(np.abs(iso["Teff"] - Teff_obs))
    model = iso.iloc[idx]

    chi2_Teff = ((Teff_obs - model["Teff"]) / 50.0)**2

    bands = ["G_mag", "BP_mag", "RP_mag", "J_mag", "H_mag", "K_mag"]
    chi2_phot = 0
    for band in bands:
        if band in model and band[0] in phot_obs:
            obs_val = phot_obs[band[0]]
            chi2_phot += ((obs_val - float(model[band])) / 0.05)**2

    return chi2_Teff + chi2_phot

In [7]:
from scipy.optimize import minimize

mist = MIST_Isochrone()

res = minimize(
    chi2_star,
    x0=[9.5],
    args=(known_stars["HD209458"], mist),
    bounds=[(6, 10.2)],
    options={"disp": True}
)

# print the best-fit model
best_age_log = res.x[0]
best_age_gyr = 10**best_age_log / 1e9

# find nearest model to best-fit age
iso_best = mist.isochrone(age=best_age_log, feh=known_stars["HD209458"]["FeH"])
idx_best = np.argmin(np.abs(iso_best["Teff"] - known_stars["HD209458"]["Teff"]))
model_best = iso_best.iloc[idx_best]

print("=== Best-fit Result ===")
print(f"Best-fit log(age/yr): {best_age_log:.3f}")
print(f"Best-fit age [Gyr]: {best_age_gyr:.2f}")
print(f"Nearest model: Mass={model_best['initial_mass']:.3f}, Teff={model_best['Teff']:.0f}, G={model_best['G_mag']:.3f}")

RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            1     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.80952D+04    |proj g|=  7.00000D-01

At iterate    1    f=  1.80815D+04    |proj g|=  6.99505D-01



 Bad direction in the line search;
   refresh the lbfgs memory and restart the iteration.



           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
    1      2     53      3     0     1   6.995D-01   1.808D+04
  F =   18081.460813390913     

ABNORMAL_TERMINATION_IN_LNSRCH                              
=== Best-fit Result ===
Best-fit log(age/yr): 9.500
Best-fit age [Gyr]: 3.17
Nearest model: Mass=1.086, Teff=6059, G=4.187



 Line search cannot locate an adequate point after MAXLS
  function and gradient evaluations.
  Previous x, f and g restored.
 Possible causes: 1 error in function or gradient evaluation;
                  2 rounding error dominate computation.


In [8]:
# comparison of raw photometry vs Teff/[Fe/H] using the Sun:
sun = {
    "phot": {
        "G":   (4.67, 0.01),
        "BP":  (4.83, 0.01),
        "RP":  (4.04, 0.01),
        "J":   (3.57, 0.02),
        "H":   (3.25, 0.02),
        "K":   (3.22, 0.02),
    },
    "spec": {
        "Teff": (5777, 50),
        "feh":  (0.00, 0.05),
    }
}

# photometry only Chi^2
def chi2_phot_only(age_log, star, mist):
    age_log = float(age_log)

    iso = mist.isochrone(age=age_log, feh=0.0)

    chi2_min = np.inf

    for _, model in iso.iterrows():
        chi2 = 0.0

        for band, (obs, err) in star["phot"].items():
            model_mag = model[f"{band}_mag"]
            chi2 += ((obs - model_mag) / err) ** 2

        chi2_min = min(chi2_min, chi2)

    return chi2_min

# Teff and [Fe/H] Chi^2
def chi2_spec_only(age_log, star, mist):
    age_log = float(age_log)

    iso = mist.isochrone(age=age_log, feh=star["spec"]["feh"][0])

    chi2_min = np.inf

    for _, model in iso.iterrows():
        chi2 = 0.0

        Teff_obs, Teff_err = star["spec"]["Teff"]
        chi2 += ((Teff_obs - model["Teff"]) / Teff_err) ** 2

        chi2_min = min(chi2_min, chi2)

    return chi2_min

res_phot = minimize(
    chi2_phot_only,
    x0=[9.5],
    args=(sun, mist),
    bounds=[(6.5, 10.2)]
)

age_phot = 10**res_phot.x[0] / 1e9

res_spec = minimize(
    chi2_spec_only,
    x0=[9.5],
    args=(sun, mist),
    bounds=[(6.5, 10.2)]
)

age_spec = 10**res_spec.x[0] / 1e9

print("=== Sun Age Test ===")
print(f"Photometry-only age: {age_phot:.2f} Gyr")
print(f"Teff + [Fe/H] age:   {age_spec:.2f} Gyr")

/var/folders/q7/mvl13nnj2v58d6nv9yfjq7680000gn/T/ipykernel_32471/928045321.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  age_log = float(age_log)
/var/folders/q7/mvl13nnj2v58d6nv9yfjq7680000gn/T/ipykernel_32471/928045321.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  age_log = float(age_log)
/var/folders/q7/mvl13nnj2v58d6nv9yfjq7680000gn/T/ipykernel_32471/928045321.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  age_log = float(age_log)
/var/folde

=== Sun Age Test ===
Photometry-only age: 5.32 Gyr
Teff + [Fe/H] age:   1.43 Gyr


/var/folders/q7/mvl13nnj2v58d6nv9yfjq7680000gn/T/ipykernel_32471/928045321.py:38: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  age_log = float(age_log)


In [9]:
known_stars = {
    "Sun": {
        "Teff": 5772,
        "Teff_err": 20,
        "feh": 0.0,
        "feh_err": 0.05,
        "G": 4.67,
        "G_err": 0.02,
    },
    "HD209458": {
        "Teff": 6075,
        "Teff_err": 50,
        "feh": 0.02,
        "feh_err": 0.05,
        "G": 7.65,
        "G_err": 0.02,
    },
    "APCol": {
        "Teff": 3470,
        "Teff_err": 100,
        "feh": 0.0,
        "feh_err": 0.2,
        "G": 8.64,
        "G_err": 0.05,
    },
}

In [10]:
import numpy as np

def chi2_age(age_log, star, mist, mode="phot"):
    iso = mist.isochrone(age=age_log, feh=star["feh"])

    chi2_min = np.inf

    for _, m in iso.iterrows():
        chi2 = 0.0

        if mode in ["teff", "teff+G"]:
            chi2 += ((m["Teff"] - star["Teff"]) / star["Teff_err"])**2

        if mode in ["phot", "teff+G"]:
            chi2 += ((m["G_mag"] - star["G"]) / star["G_err"])**2

        if chi2 < chi2_min:
            chi2_min = chi2

    return chi2_min

from scipy.optimize import minimize

def fit_age(star, mist, mode):
    res = minimize(
        chi2_age,
        x0=[9.5],
        args=(star, mist, mode),
        bounds=[(6.0, 10.2)],
        method="L-BFGS-B"
    )

    age_log = res.x[0]
    age_gyr = 10**age_log / 1e9

    return age_log, age_gyr

mist = MIST_Isochrone()

for name, star in known_stars.items():
    print(f"\n=== {name} ===")

    logA_phot, A_phot = fit_age(star, mist, mode="phot")
    print(f"Photometry-only age: {A_phot:.2f} Gyr")

    logA_teff, A_teff = fit_age(star, mist, mode="teff")
    print(f"Teff + [Fe/H] age:   {A_teff:.2f} Gyr")

    logA_combo, A_combo = fit_age(star, mist, mode="teff+G")
    print(f"Teff + [Fe/H] + G:   {A_combo:.2f} Gyr")


=== Sun ===
Photometry-only age: 15.85 Gyr
Teff + [Fe/H] age:   0.61 Gyr
Teff + [Fe/H] + G:   3.15 Gyr

=== HD209458 ===
Photometry-only age: 3.12 Gyr
Teff + [Fe/H] age:   5.35 Gyr
Teff + [Fe/H] + G:   3.18 Gyr

=== APCol ===
Photometry-only age: 10.22 Gyr
Teff + [Fe/H] age:   2.81 Gyr
Teff + [Fe/H] + G:   0.06 Gyr
